# CodeBuddy Agent
AWS Bedrock 기반 GitHub PR 자동 리뷰 Agent

In [ ]:
# 1. 패키지 설치
!pip install boto3 pygithub -q

In [ ]:
# 2. 환경 설정
import os, io, json, time, boto3, zipfile, shutil, subprocess
import botocore.exceptions
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-northeast-2'

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
agent_id = userdata.get('AGENT_ID')
alias_id = userdata.get('AGENT_ALIAS_ID')

sts = boto3.client('sts')
lambda_client = boto3.client('lambda')
bedrock_agent = boto3.client('bedrock-agent')
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

account_id = sts.get_caller_identity()['Account']
lambda_role_arn = f'arn:aws:iam::{account_id}:role/CodeBuddy-Lambda-Role'
print(f'✅ 환경 설정 완료 (Account: {account_id})')

In [ ]:
# 3. Lambda 코드 정의
LAMBDA_PR_CODE = open('lambda/github_pr.py').read() if os.path.exists('lambda/github_pr.py') else '''
import json, os, logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    try:
        params = {p["name"]: p["value"] for p in event.get("parameters", [])}
        owner, repo_name, pr_number = params.get("owner"), params.get("repo"), params.get("pr_number")
        if not all([owner, repo_name, pr_number]):
            raise ValueError("Missing required parameters")

        g = Github(os.environ["GITHUB_TOKEN"])
        repo = g.get_repo(f"{owner}/{repo_name}")
        pr = repo.get_pull(int(pr_number))

        files = pr.get_files()
        diff_content = [{"filename": f.filename, "patch": f.patch or "", "additions": f.additions, "deletions": f.deletions} for f in files]

        return build_response(event, 200, {
            "title": pr.title, "body": pr.body or "", "state": pr.state,
            "author": pr.user.login, "created_at": pr.created_at.isoformat(),
            "changed_files": pr.changed_files, "additions": pr.additions,
            "deletions": pr.deletions, "diff": diff_content
        })
    except GithubException as e:
        return build_response(event, 404, {"error": str(e)})
    except Exception as e:
        return build_response(event, 500, {"error": str(e)})

def build_response(event, status, body):
    return {"messageVersion": "1.0", "response": {"actionGroup": event.get("actionGroup"),
        "apiPath": event.get("apiPath"), "httpMethod": event.get("httpMethod"),
        "httpStatusCode": status, "responseBody": {"application/json": {"body": body}}}}
'''

LAMBDA_LIST_REPOS_CODE = '''
import json, os, logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    try:
        params = {p["name"]: p["value"] for p in event.get("parameters", [])}
        owner = params.get("owner")
        repo_type = params.get("type", "all")
        if not owner:
            raise ValueError("Missing required parameter: owner")

        g = Github(os.environ["GITHUB_TOKEN"])
        try:
            user = g.get_user(owner)
        except GithubException:
            user = g.get_organization(owner)

        repos = [{"name": r.name, "full_name": r.full_name, "description": r.description,
                  "html_url": r.html_url, "private": r.private, "fork": r.fork, "language": r.language}
                 for r in user.get_repos(type=repo_type)]

        return build_response(event, 200, repos)
    except GithubException as e:
        return build_response(event, 404, {"error": str(e)})
    except Exception as e:
        return build_response(event, 500, {"error": str(e)})

def build_response(event, status, body):
    return {"messageVersion": "1.0", "response": {"actionGroup": event.get("actionGroup"),
        "apiPath": event.get("apiPath"), "httpMethod": event.get("httpMethod"),
        "httpStatusCode": status, "responseBody": {"application/json": {"body": body}}}}
'''
print('✅ Lambda 코드 정의 완료')

In [ ]:
# 4. Lambda 배포 패키지 생성 및 배포
def make_zip(code, temp_dir):
    os.makedirs(temp_dir, exist_ok=True)
    with open(os.path.join(temp_dir, 'index.py'), 'w') as f:
        f.write(code)
    subprocess.check_call(['pip', 'install', 'PyGithub', '-t', temp_dir, '-q'])
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, temp_dir))
    buf.seek(0)
    shutil.rmtree(temp_dir)
    return buf.getvalue()

def deploy_lambda(function_name, code, temp_dir):
    zip_bytes = make_zip(code, temp_dir)
    try:
        res = lambda_client.create_function(
            FunctionName=function_name, Runtime='python3.12',
            Role=lambda_role_arn, Handler='index.handler',
            Code={'ZipFile': zip_bytes},
            Environment={'Variables': {'GITHUB_TOKEN': GITHUB_TOKEN}},
            Timeout=30
        )
        print(f'✅ Lambda 생성: {res["FunctionArn"]}')
        return res['FunctionArn']
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        print(f'✅ Lambda 업데이트: {function_name}')
        return f'arn:aws:lambda:ap-northeast-2:{account_id}:function:{function_name}'

pr_lambda_arn = deploy_lambda('codebuddy-github-pr', LAMBDA_PR_CODE, './pkg_pr')
list_repos_lambda_arn = deploy_lambda('codebuddy-list-repos', LAMBDA_LIST_REPOS_CODE, './pkg_repos')

In [ ]:
# 5. OpenAPI 스키마 정의
PR_SCHEMA = {
    'openapi': '3.0.0',
    'info': {'title': 'GitHub PR Tools', 'version': '1.0.0'},
    'paths': {
        '/pr': {'get': {
            'operationId': 'get_github_pr',
            'summary': 'Get GitHub Pull Request details and code diff',
            'parameters': [
                {'name': 'owner', 'in': 'query', 'required': True, 'schema': {'type': 'string'}},
                {'name': 'repo', 'in': 'query', 'required': True, 'schema': {'type': 'string'}},
                {'name': 'pr_number', 'in': 'query', 'required': True, 'schema': {'type': 'integer'}}
            ],
            'responses': {'200': {'description': 'PR details and diff'}}
        }}
    }
}

REPOS_SCHEMA = {
    'openapi': '3.0.0',
    'info': {'title': 'GitHub Repo Tools', 'version': '1.0.0'},
    'paths': {
        '/repos': {'get': {
            'operationId': 'list_repositories',
            'summary': 'List GitHub repositories',
            'parameters': [
                {'name': 'owner', 'in': 'query', 'required': True, 'schema': {'type': 'string'}},
                {'name': 'type', 'in': 'query', 'required': False, 'schema': {'type': 'string', 'enum': ['all', 'public', 'private']}}
            ],
            'responses': {'200': {'description': 'Repository list'}}
        }}
    }
}
print('✅ 스키마 정의 완료')

In [ ]:
# 6. Agent 설정
def prepare_agent(agent_id, max_wait=60):
    bedrock_agent.prepare_agent(agentId=agent_id)
    print('⏳ Agent Prepare 중...')
    for i in range(max_wait):
        status = bedrock_agent.get_agent(agentId=agent_id)['agent']['agentStatus']
        if status == 'PREPARED':
            print(f'✅ Prepare 완료 ({i+1}초)')
            return True
        elif status == 'FAILED':
            print('❌ Prepare 실패')
            return False
        time.sleep(1)

def add_permission_safe(function_name, statement_id):
    try:
        lambda_client.add_permission(
            FunctionName=function_name, StatementId=statement_id,
            Action='lambda:InvokeFunction', Principal='bedrock.amazonaws.com',
            SourceArn=f'arn:aws:bedrock:ap-northeast-2:{account_id}:agent/{agent_id}'
        )
        print(f'✅ {function_name} 권한 추가')
    except lambda_client.exceptions.ResourceConflictException:
        print(f'⚠️ {function_name} 권한 이미 존재')

def create_action_group_safe(name, lambda_arn, schema):
    try:
        bedrock_agent.create_agent_action_group(
            agentId=agent_id, agentVersion='DRAFT',
            actionGroupName=name,
            actionGroupExecutor={'lambda': lambda_arn},
            apiSchema={'payload': json.dumps(schema)},
            actionGroupState='ENABLED'
        )
        print(f'✅ Action Group 생성: {name}')
    except Exception as e:
        print(f'⚠️ {name}: {e}')

add_permission_safe('codebuddy-github-pr', 'bedrock-agent-invoke-pr')
add_permission_safe('codebuddy-list-repos', 'bedrock-agent-invoke-repos')
create_action_group_safe('GitHubPRTools', pr_lambda_arn, PR_SCHEMA)
create_action_group_safe('GitHubRepoTools', list_repos_lambda_arn, REPOS_SCHEMA)

agent_info = bedrock_agent.get_agent(agentId=agent_id)['agent']
bedrock_agent.update_agent(
    agentId=agent_id,
    agentName=agent_info['agentName'],
    agentResourceRoleArn=agent_info['agentResourceRoleArn'],
    foundationModel=agent_info['foundationModel'],
    instruction="""당신은 시니어 개발자이자 코드 리뷰 전문가 CodeBuddy입니다.

## 역할
- Python, JavaScript, Java 코드 리뷰를 수행합니다.
- 버그, 보안 취약점, 스타일 위반을 찾습니다.

## 사용 가능한 도구
1. get_github_pr: GitHub PR 상세 정보 및 코드 diff 조회
   - 파라미터: owner, repo, pr_number
   - PR 정보 요청 시 반드시 호출하세요.
   - diff 내용을 분석하여 코드 리뷰를 제공하세요.

2. list_repositories: GitHub 저장소 목록 조회
   - 파라미터: owner, type(선택: all/public/private)
   - 저장소 목록 요청 시 반드시 호출하세요.

## 규칙
- 도구로 해결 가능한 요청은 반드시 도구를 호출하세요.
- PR 리뷰 시 diff 내용을 분석하여 구체적인 피드백을 제공하세요.
- 필수 파라미터가 없으면 사용자에게 물어보세요.

## 출력 형식
🔴 높은 심각도: 문제 설명 + 수정 코드
🟡 중간 심각도: 개선 제안
🟢 낮은 심각도: 스타일 권장사항"""
)
print('✅ Instruction 업데이트 완료')
prepare_agent(agent_id)

In [ ]:
# 7. Agent 테스트
def invoke_agent(prompt, session_id='test-001'):
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=True
    )
    print(f'\n🤖 [{prompt}]')
    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'), end='')
        elif 'trace' in event:
            orch = event['trace']['trace'].get('orchestrationTrace', {})
            if 'invocationInput' in orch:
                print(f'\n🔧 Tool: {orch["invocationInput"]["actionGroupInvocationInput"]["actionGroupName"]}')

# PR 리뷰 테스트
invoke_agent('cksdud1202a/kt-cloud-basic 저장소의 PR #1 코드를 리뷰해줘', 'session-review')

In [ ]:
# 저장소 목록 테스트
invoke_agent('cksdud1202a의 GitHub 저장소 목록을 보여줘', 'session-repos')

In [ ]:
# 8. Slack 알림
import requests

SLACK_WEBHOOK = userdata.get('SLACK_WEBHOOK_URL')

def send_slack(pr_url, summary):
    message = {
        'text': f'🤖 *CodeBuddy 리뷰 완료*\nPR: {pr_url}\n요약: {summary[:300]}...'
    }
    res = requests.post(SLACK_WEBHOOK, json=message)
    print(f'✅ Slack 전송 완료 (status: {res.status_code})')

# PR 리뷰 후 Slack 알림
pr_url = 'https://github.com/cksdud1202a/kt-cloud-basic/pull/1'
review_result = '코드 리뷰 완료. 자세한 내용은 PR 댓글을 확인하세요.'
send_slack(pr_url, review_result)